In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from statsmodels.stats.multitest import multipletests
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import itertools
from matplotlib.lines import Line2D
import mpmath

### Simulation répartition aléatoire du nb de mutations

In [ ]:
# Define parameters
num_positions = 316  # Total positions
total_mutations = 79  # Total number of mutations
num_simulations = 10000  # Number of simulations

# Weighted probabilities based on observed substitution rates
#codon_position_weights = [0.25, 0.15, 0.60]  # 1st, 2nd, 3rd

# Pick a codon at random, then mutate one position based on the weights
#gene_codons = 312 // 3
#mutation_positions = []

#for _ in range(n_mutations):
#    codon_index = np.random.randint(0, gene_codons)
#    codon_pos = np.random.choice([0, 1, 2], p=codon_position_weights)
#    mutation_pos = codon_index * 3 + codon_pos
#    mutation_positions.append(mutation_pos)


# Load sequence coverage data from CSV
coverage_df = pd.read_csv("Path_to_weightFile_positions")
sequence_coverage = coverage_df["Coverage"].values

# Normalize coverage to get probabilities
mutation_probabilities = sequence_coverage / sequence_coverage.sum()

# Generate all simulations at once using weighted probabilities
simulations = np.array([
    np.random.multinomial(total_mutations, mutation_probabilities) for _ in range(num_simulations)
]).T

# Create DataFrame
positions = np.arange(1, num_positions + 1)
columns = [f"Simulation_{i+1}" for i in range(num_simulations)]
mutation_df = pd.DataFrame(simulations, columns=columns)
mutation_df.insert(0, "Position", positions)
mutation_df.insert(1, "Coverage", sequence_coverage)

# Calculate mean and standard error for each position
#mutation_df['Mean_Mutations'] = mutation_df.iloc[:, 2:].mean(axis=1)
#mutation_df['Std_Error'] = mutation_df.iloc[:, 2:].sem(axis=1)

# Calculate normalized sum of mutations per position
#mutation_df['Normalized_Sum'] = mutation_df.iloc[:, 2:].sum(axis=1) / num_simulations

# Save to CSV (optional)
mutation_df.to_csv("Path_to_folder/random_mutations_10000_simulations_TRA.csv", index=False)

# Display first few rows
mutation_df.head()


## Calculate probabililty

In [ ]:
# Load real data
real_df = pd.read_csv("Path_to_folder/GeneV_mutations_positions_TRA.csv")  # Adjust path
real_mutations = real_df["Total TRA"].values

# Load simulated mutation data
sim_df = pd.read_csv("Path_to_folder/random_mutations_10000_simulations_TRA.csv")
# Get only the simulation columns as a NumPy array
simulation_data = sim_df.filter(like="Simulation_").values


# Sanity check
num_positions=316
assert len(real_mutations) == num_positions, "Mismatch in number of positions"

p_values = []
for i in range(num_positions):
    simulated_counts = simulation_data[i]
    real_count = real_mutations[i]
    p = np.mean(simulated_counts >= real_count)
    p_values.append(p)

# Add real mutations and p-values to the mutation_df (which you can reload or reuse)
mutation_df = sim_df.copy()
mutation_df['Real_Mutations'] = real_mutations
mutation_df['P_Value'] = p_values

#Multiple testing correction
mutation_df['Adjusted_P'] = multipletests(mutation_df['P_Value'], method='fdr_bh')[1]

# Save the result
mutation_df.to_csv("Path_to_folder/mutation_comparison_with_pvalues_10000sim_TRA.csv", index=False)


## Statistics on region

In [ ]:
# Load real data
real_df = pd.read_csv("Path_to_folder/GeneV_mutations_positions_TRA.csv")  # Adjust path
real_mutations = real_df["Total TRA"].values

# Load simulated mutation data
sim_df = pd.read_csv("Path_to_folder/random_mutations_10000_simulations_TRA.csv")
# Get only the simulation columns as a NumPy array
simulation_data = sim_df.filter(like="Simulation_").values

# Set the precision (number of decimal places)
mpmath.mp.dps = 50  # 50 decimal places

##Define region
# List of tuples: (region_name, start_pos, end_pos)
regions = [
    ("Region_1", 0, 80),
    ("Region_2", 81, 116),
    ("Region_3", 117, 167),
    ("Region_4", 168, 197),
    ("Region_5", 198, 315)
]

## Compute Real and Simulated Mutation Totals per Region
region_results = []

for name, start, end in regions:
    indices = list(range(start, end + 1))
    region_length = len(indices)

    # Real data
    real_sum = real_mutations[indices].sum()

    # Simulations
    sim_sums = simulation_data[indices, :].sum(axis=0)

    # Convert to mpmath format
    real_sum_mp = mpmath.mpf(str(real_sum))
    sim_sums_mp = [mpmath.mpf(str(s)) for s in sim_sums]

    # Calculate p-value using higher precision
    count = sum(1 for s in sim_sums_mp if s >= real_sum_mp)
    p_val = float(count / len(sim_sums_mp))
    
    # Empirical p-value
    #p_val = np.mean(sim_sums >= real_sum)


    region_results.append({
        "Region": name,
        "Start": start,
        "End": end,
        "Length": region_length,
        "Real_Mutations": real_sum,
        "Mean_Simulated": sim_sums.mean(),
        "P_Value": p_val,
        "Simulated_Distribution": sim_sums  # keep for later pairwise tests
    })

##Multiple Testing Correction

p_vals = [r["P_Value"] for r in region_results]
adjusted_p = multipletests(p_vals, method='fdr_bh')[1]

for i, adj_p in enumerate(adjusted_p):
    region_results[i]["Adjusted_P"] = adj_p


# Format p-values in scientific notation
#for result in region_results:
#    result["P_Value_Scientific"] = f"{result['P_Value']:.2e}"
#    result["Adjusted_P_Scientific"] = f"{result['Adjusted_P']:.2e}"

# --- Save region vs simulation results ---
region_df = pd.DataFrame([{
    "Region": r["Region"],
    "Start": r["Start"],
    "End": r["End"],
    "Length": r["Length"],
    "Real_Mutations": r["Real_Mutations"],
    "Mean_Simulated": r["Mean_Simulated"],
    "P_Value": r["P_Value"],
    "Adjusted_P": r["Adjusted_P"]
} for r in region_results])

region_df.to_csv("Path_to_folder/region_vs_simulation_results_TRA_10000.csv", index=False)

# --- Region vs region empirical comparison ---
def empirical_region_comparison(region_a, region_b):
    sim_a = region_a["Simulated_Distribution"] / region_a["Length"]
    sim_b = region_b["Simulated_Distribution"] / region_b["Length"]
    
    real_density_a = region_a["Real_Mutations"] / region_a["Length"]
    real_density_b = region_b["Real_Mutations"] / region_b["Length"]
    real_diff = real_density_a - real_density_b

    sim_diff = sim_a - sim_b
    p_empirical = np.mean(np.abs(sim_diff) >= np.abs(real_diff))

    return real_density_a, real_density_b, real_diff, p_empirical

# --- All pairwise comparisons ---
pairwise_results = []
for region_a, region_b in itertools.combinations(region_results, 2):
    dens_a, dens_b, diff, pval = empirical_region_comparison(region_a, region_b)
    pairwise_results.append({
        "Region_A": region_a["Region"],
        "Region_B": region_b["Region"],
        "Density_A": dens_a,
        "Density_B": dens_b,
        "Density_Difference": diff,
        "Empirical_P": pval
    })

# Extract p-values for multiple testing correction
p_vals_pairwise = [r["Empirical_P"] for r in pairwise_results]
adjusted_p_pairwise = multipletests(p_vals_pairwise, method='fdr_bh')[1]

# Add adjusted p-values to the pairwise results
for i, adj_p in enumerate(adjusted_p_pairwise):
    pairwise_results[i]["Adjusted_P"] = adj_p


# --- Save region vs region comparison ---
pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df.to_csv("Path_to_folder/region_vs_region_comparison_TRA_10000.csv", index=False)

print("Analysis complete. Results saved to:")
print("region_vs_simulation_results.csv")
print("region_vs_region_comparison.csv")


## Visualisation with simulation and mutations

In [ ]:

file_path_1 = "Path_to_folder/WeightPositions_TRA_sim.csv"  # Update this with your file path
file_path_2 = "Path_to_folder/GeneV_mutations_positions_TRA.csv"  

df1 = pd.read_csv(file_path_1)
df2 = pd.read_csv(file_path_2)

# Filter positions where Coverage is not zero
filtered_df1 = df1[df1['Coverage'] != 0]
positions = filtered_df1['Positions']

# Merge df2 with filtered_df1 to align indices
merged_df = pd.merge(df2, filtered_df1[['Positions']], on='Positions', how='inner')

#positions = df1['Position']
silent_tra = merged_df['Silent TRA']
aa_change_tra = merged_df['AA change TRA']
# Liste des colonnes UMI Ref et UMI Mut
umi_ref_cols = merged_df[['UMI Ref TRA 1', 'UMI Ref TRA 2', 'UMI Ref TRA 3', 'UMI Ref TRA 4']] 
umi_mut_cols = merged_df[['UMI Mut TRA 1', 'UMI Mut TRA 2', 'UMI Mut TRA 3', 'UMI Mut TRA 4']] 

# Define position ranges
ranges = [(1, 117), (118, 229), (230, 316)]
bar_width = 0.5
circle_offset = 4  # vertical offset
circle_size = 0.3  # pie size

# Create figure and custom GridSpec layout
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(8, 1, height_ratios=[1, 0.2, 1, 0.2, 0.2, 0.01, 0.02, 0.5], hspace=0.3)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[2])
ax3_top = fig.add_subplot(gs[4:6])        # No more sharex
ax3_bottom = fig.add_subplot(gs[6:])      # No more sharex
axes = [ax1, ax2]

# Y-axis limits
ax1.set_ylim(0, 16)
ax2.set_ylim(0, 16)
ax3_top.set_ylim(24, 28)
ax3_bottom.set_ylim(0, 15)

# Set y-ticks manually
ax3_top.set_yticks([25, 27])

for i, (start, end) in enumerate(ranges):
    mask = (merged_df['Positions'] >= start) & (merged_df['Positions'] <= end)
    subset = merged_df[mask].reset_index(drop=True)
    x_indexes = np.arange(len(subset))

    silent_vals = subset['Silent TRA']
    aa_vals = subset['AA change TRA']


    if i < 2:
        ax = axes[i]
        ax.bar(x_indexes, silent_vals, width=bar_width, color='navy', alpha=0.6)
        ax.bar(x_indexes, aa_vals, width=bar_width, color='red', alpha=0.6, bottom=silent_vals)

        for j, x in enumerate(x_indexes):
            for k, (ref_col, mut_col) in enumerate(zip(umi_ref_cols, umi_mut_cols)):
                r, m = subset[ref_col][j], subset[mut_col][j]
                total = r + m
                if total > 0 and r > 0:
                    y_pos = silent_vals[j] + aa_vals[j] + circle_offset * (k + 1)
                    axins = inset_axes(ax, width=circle_size, height=circle_size, loc='center',
                                       bbox_to_anchor=(x, y_pos),
                                       bbox_transform=ax.transData,
                                       borderpad=0)
                    axins.pie([r, m], colors=['green', 'orange'], startangle=90)
                    axins.set_aspect('equal')

                    # Add text annotation for Mut UMI
                    axins.text(0, 0, str(m), ha='center', va='center', fontsize=8, color='black')

                    axins.axis('off')

        ax.set_xlim(-0.5, len(subset) - 0.5)
        ax.set_xticks(x_indexes[::2])
        ax.set_xticklabels(subset['Positions'][::2], rotation=45, fontsize=8)
        ax.set_ylabel("Count")

    else:
        for ax in (ax3_bottom, ax3_top):
            ax.bar(x_indexes, silent_vals, width=bar_width, color='navy', alpha=0.6)
            ax.bar(x_indexes, aa_vals, width=bar_width, color='red', alpha=0.6, bottom=silent_vals)

        for j, x in enumerate(x_indexes):
            for k, (ref_col, mut_col) in enumerate(zip(umi_ref_cols, umi_mut_cols)):
                r, m = subset[ref_col][j], subset[mut_col][j]
                total = r + m
                height = silent_vals[j] + aa_vals[j]

                if total > 0 and r > 0:
                    if height <= 5:
                        ax_use = ax3_bottom
                        y_pos = height + circle_offset * (k + 1)
                    elif height >= 20:
                        ax_use = ax3_top
                        y_pos = height - 2 * (k + 1)
                    else:
                        continue

                    axins = inset_axes(ax_use, width=circle_size, height=circle_size, loc='center',
                                       bbox_to_anchor=(x, y_pos),
                                       bbox_transform=ax_use.transData,
                                       borderpad=0)
                    axins.pie([r, m], colors=['green', 'orange'], startangle=90)
                    axins.set_aspect('equal')
                    axins.axis('off')


        # Set x-axis only for bottom subplot
        ax3_bottom.set_xlim(-0.5, len(subset) - 0.5)
        ax3_bottom.set_xticks(x_indexes[::2])
        ax3_bottom.set_xticklabels(subset['Positions'][::2], rotation=45, fontsize=8)
        ax3_bottom.set_ylabel("Count")

        ax3_top.set_xlim(-0.5, len(subset) - 0.5)
        ax3_top.set_ylabel("Count")
        ax3_top.set_ylabel("")
        ax3_top.spines.bottom.set_visible(False)
        ax3_bottom.spines.top.set_visible(False)
        ax3_top.tick_params(labeltop=False, top =False)
        ax3_top.tick_params(labelbottom=False, bottom=False)
        #ax3_top.xaxis.tick_top()
        ax3_bottom.xaxis.tick_bottom()

        # Draw break marks
        d = .5
        kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                      linestyle="none", color='k', mec='k', mew=1, clip_on=False)
        ax3_top.plot([0, 1], [0, 0], transform=ax3_top.transAxes, **kwargs)
        ax3_bottom.plot([0, 1], [1, 1], transform=ax3_bottom.transAxes, **kwargs)

# Add x-label only to bottom-most plot
ax3_bottom.set_xlabel("Positions")

# Legend
handles = [
    plt.Rectangle((0, 0), 1, 1, color='navy', alpha=0.6, label='Silent TRA'),
    plt.Rectangle((0, 0), 1, 1, color='red', alpha=0.6, label='AA Change TRA'),
    patches.Wedge((0, 0), 0.1, 0, 180, color='green', alpha=0.6, label='Ref UMI %'),
    patches.Wedge((0, 0), 0.1, 180, 360, color='orange', alpha=0.6, label='Mut UMI %')
]

fig.legend(handles=handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05))

# Save and show
#plt.tight_layout()
plt.subplots_adjust(hspace=0.5)
plt.savefig("Path_to_folder/histogram_plot_TRA.svg", format="svg", dpi=500, bbox_inches='tight')
plt.show()


### Visualisation with only mutations plus probability plus UMIs

In [ ]:
file_path_1 = "Path_to_folder/mutation_comparison_with_pvalues_10000sim.csv"  # Update this with your file path
file_path_2 = "Path_to_folder/GeneV_mutations_positions_TRA.csv"  

df1 = pd.read_csv(file_path_1)
df2 = pd.read_csv(file_path_2)

positions = df1['Position']
silent_trb = df2['Silent TRA']
aa_change_trb = df2['AA change TRA']
p_values = df1['P_Value']

# Define position ranges
ranges = [(1, 117), (118, 229), (230, 316)]
bar_width = 0.4  # Width of the bars


# Create figure and axis
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=False) 

y_limit = (0, 8)

for i, (start, end) in enumerate(ranges):
    mask = (positions >= start) & (positions <= end)  # Filter out positions with mutations_trb = 0
    subset_positions = positions[mask]

    if subset_positions.empty:
        # Skip the plot if there are no positions in the subset
        continue

    x_indexes = np.arange(len(subset_positions))  # Create index for each position

    # Stack Silent TRA and AA Change TRA
    bars_silent = axes[i].bar(x_indexes, silent_trb[mask], width=bar_width, color='navy', alpha=0.6, label='Silent TRA')
    bars_aa_change = axes[i].bar(x_indexes, aa_change_trb[mask], width=bar_width, color='red', alpha=0.6, bottom=silent_tra[mask], label='AA Change TRA')

    # Add stars for p-values < 0.05
    for bar, p_value in zip(bars_silent + bars_aa_change + bars_sim, p_values[mask]):
        if p_value < 0.05:
            axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, '*', ha='center', va='bottom', fontsize=12, color='black')

    axes[i].set_xlim(-0.5, len(subset_positions) - 0.5)
    axes[i].set_ylim(y_limit)
    axes[i].set_ylabel("Count")
    axes[i].set_xticks(x_indexes[::2])
    axes[i].set_xticklabels(subset_positions.iloc[::2], rotation=45, fontsize=8)  # Reduce x-axis label size

# Set xlabel for the last subplot
axes[-1].set_xlabel("Positions")

# Add a single legend below all subplots
handles = [
    plt.Rectangle((0, 0), 1, 1, color='navy', alpha=0.6, label='Silent TRA'),
    plt.Rectangle((0, 0), 1, 1, color='red', alpha=0.6, label='AA Change TRA')
]

fig.legend(handles=handles, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.1))

# Adjust layout and save plot
plt.tight_layout()
plt.subplots_adjust(hspace=0.5)  # Adjust space between subplots
plt.savefig("Path_to_folder/Plot_mutations_TRA.svg", format="svg", dpi=500, bbox_inches='tight')
plt.show()